# Gemma 4 Two-Video Naturalness Demo

Self-contained Colab-friendly notebook for checking whether videos look like real natural footage or AI-generated content.

Default setup uses Google Drive paths under `MyDrive/Colab Notebooks`.

## Setup

1. Use a GPU runtime.
2. Make sure you have accepted the Gemma license on Hugging Face.
3. Provide `HF_TOKEN` through an environment variable or enter it when prompted.
4. Update `VIDEO_PATHS` to point at your own `.mp4` files.


In [ ]:
%pip install -q -U transformers accelerate bitsandbytes pillow==11.3.0 huggingface_hub opencv-python

In [ ]:
import getpass
import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

HF_TOKEN = os.getenv("HF_TOKEN", "").strip()
if not HF_TOKEN:
    HF_TOKEN = getpass.getpass("HF_TOKEN (leave blank if already logged in): ").strip()

if HF_TOKEN:
    from huggingface_hub import login
    login(HF_TOKEN, add_to_git_credential=False)

import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no gpu")


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:

import json
import re
from collections import Counter
from pathlib import Path

import cv2
import torch
from transformers import BitsAndBytesConfig, GenerationConfig, pipeline

VIDEO_PATHS = [
    "demo_videos/good_video.mp4",
    "demo_videos/bad_video.mp4",
]
MODEL_ID = "google/gemma-4-E2B-it"
FRAME_LIMIT = 4
FRAME_INTERVAL = 14
FRAME_MAX_EDGE = 448
MAX_NEW_TOKENS = 192
RETRY_MAX_NEW_TOKENS = 128
CONSENSUS_RUNS = 3
CONSENSUS_MAX_ATTEMPTS = 5


REALISM_PRIORITY = {
    "high_realism": 2,
    "medium_realism": 1,
    "low_realism": 0,
}

print("Update VIDEO_PATHS to match your environment if needed.")
for video_path in VIDEO_PATHS:
    print(video_path, Path(video_path).exists())

def clamp(value: float, lo: float = 0.0, hi: float = 10.0) -> float:
    return round(max(lo, min(hi, value)), 3)

def collect_video_metadata(video_path: str) -> dict:
    capture = cv2.VideoCapture(video_path)
    frame_count = int(capture.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    fps = float(capture.get(cv2.CAP_PROP_FPS) or 0.0)
    width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH) or 0)
    height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT) or 0)
    capture.release()
    duration_seconds = round(frame_count / fps, 2) if fps > 0 else 0.0
    return {
        "frame_count": frame_count,
        "fps": round(fps, 2),
        "duration_seconds": duration_seconds,
        "width": width,
        "height": height,
    }

def extract_frame_evidence(video_path: str, interval: int = FRAME_INTERVAL) -> list[str]:
    capture = cv2.VideoCapture(video_path)
    output_dir = Path("frames") / Path(video_path).stem
    output_dir.mkdir(parents=True, exist_ok=True)
    frame_paths = []
    count = 0
    saved = 0
    while capture.isOpened():
        success, frame = capture.read()
        if not success:
            break
        if count % interval == 0:
            height, width = frame.shape[:2]
            max_edge = max(height, width)
            if max_edge > FRAME_MAX_EDGE:
                scale = FRAME_MAX_EDGE / max_edge
                frame = cv2.resize(
                    frame,
                    (int(width * scale), int(height * scale)),
                    interpolation=cv2.INTER_AREA,
                )
            frame_path = output_dir / f"frame_{saved:04d}.jpg"
            cv2.imwrite(str(frame_path), frame)
            frame_paths.append(str(frame_path.resolve()))
            saved += 1
        count += 1
    capture.release()
    return frame_paths[:FRAME_LIMIT]

def build_artifact_messages(video_path: str, frame_paths: list[str], metadata: dict):
    system_prompt = (
        "You are Gemma 4 acting as a strict detector of AI video artifacts. "
        "Focus only on visible synthesis failures. Return only compact JSON."
    )
    user_text = f'''Inspect this video for visible signs of AI generation.

Video:
{Path(video_path).name}

Metadata:
{json.dumps(metadata, indent=2)}

The attached images are representative frames in chronological order.

Look only for these artifact categories:
- face warping or fake-looking skin
- hand or finger errors
- mouth, food, or object interaction errors
- identity drift across frames
- motion or temporal instability
- texture flicker or geometry warping

Return only valid JSON with this exact schema. Keep every string short. Use at most 2 items per list.
{{
  "severe_artifacts": ["short phrase"],
  "moderate_artifacts": ["short phrase"],
  "artifact_summary": "max 12 words",
  "artifact_verdict": "clean|borderline|artifact_heavy"
}}'''
    content = [{"type": "text", "text": user_text}]
    for frame_path in frame_paths:
        content.append({"type": "image", "path": frame_path})
    return [
        {"role": "system", "content": [{"type": "text", "text": system_prompt}]},
        {"role": "user", "content": content},
    ]

def extract_json_object(text: str) -> dict:
    text = text.strip()
    if not text:
        raise ValueError("Model returned empty text.")

    def _sanitize(candidate: str) -> str:
        candidate = candidate.strip()
        candidate = re.sub(r"```(?:json)?", "", candidate, flags=re.IGNORECASE).strip()
        candidate = candidate.replace("```", "").strip()
        candidate = re.sub(r"<\|[^>]*\|>", "", candidate).strip()
        candidate = re.sub(r"<turn\|>", "", candidate).strip()
        if candidate.startswith('"') and ':{' not in candidate and '{' not in candidate:
            candidate = '{' + candidate + '}'
        elif ':' in candidate and not candidate.startswith('{'):
            candidate = '{' + candidate + '}'
        candidate = re.sub(r",\s*}\s*$", "}", candidate, flags=re.DOTALL)
        return candidate.strip()

    candidates = []
    fenced_match = re.search(r"```(?:json)?\s*(.*?)\s*```", text, re.DOTALL | re.IGNORECASE)
    if fenced_match:
        candidates.append(fenced_match.group(1))
    candidates.append(text)
    candidates.extend(re.findall(r"\{.*?\}", text, re.DOTALL))

    payload = None
    seen = set()
    for raw_candidate in candidates:
        candidate = _sanitize(raw_candidate)
        if not candidate or candidate in seen:
            continue
        seen.add(candidate)
        try:
            payload = json.loads(candidate)
            break
        except json.JSONDecodeError:
            continue

    if payload is None:
        sanitized = _sanitize(text)
        raise ValueError(f"No parseable JSON object found in model output: {sanitized[:400]}")
    if not isinstance(payload, dict):
        raise ValueError("Model output was not a JSON object.")
    return payload

def normalize_artifact_report(report: dict) -> dict:
    severe = report.get("severe_artifacts", []) or []
    moderate = report.get("moderate_artifacts", []) or []
    if isinstance(severe, str):
        severe = [severe]
    if isinstance(moderate, str):
        moderate = [moderate]

    def _is_placeholder(value: str) -> bool:
        placeholder_phrases = (
            "none",
            "n/a",
            "no artifact",
            "no artifacts",
            "no detected",
            "none detected",
            "no issue",
            "no issues",
            "no visible artifact",
            "no visible artifacts",
            "no obvious artifact",
            "no obvious artifacts",
            "no clear artifact",
            "no clear artifacts",
        )
        return any(phrase in value for phrase in placeholder_phrases)

    def _clean_entries(items: list[str]) -> list[str]:
        cleaned = []
        for item in items:
            value = str(item).strip().lower()
            if not value or _is_placeholder(value):
                continue
            cleaned.append(value)
        return cleaned

    def _is_explicitly_severe(value: str) -> bool:
        severe_markers = (
            "severe ",
            "major ",
            "catastrophic",
            "extreme",
            "multiple",
            "broken",
        )
        return any(marker in value for marker in severe_markers)

    raw_severe = _clean_entries(severe)
    raw_moderate = _clean_entries(moderate)
    normalized_severe = []
    normalized_moderate = []

    for value in raw_severe:
        if _is_explicitly_severe(value):
            normalized_severe.append(value)
        else:
            normalized_moderate.append(value)

    normalized_moderate.extend(raw_moderate)

    deduped_severe = []
    for value in normalized_severe:
        if value not in deduped_severe:
            deduped_severe.append(value)

    deduped_moderate = []
    for value in normalized_moderate:
        if value not in deduped_severe and value not in deduped_moderate:
            deduped_moderate.append(value)

    summary = str(report.get("artifact_summary", "")).strip()

    normalized = {
        "severe_artifacts": deduped_severe[:3],
        "moderate_artifacts": deduped_moderate[:4],
        "artifact_summary": summary,
        "artifact_verdict": "clean",
    }
    severe_count = len(normalized["severe_artifacts"])
    moderate_count = len(normalized["moderate_artifacts"])
    if severe_count >= 1 or moderate_count >= 3:
        normalized["artifact_verdict"] = "artifact_heavy"
    elif moderate_count >= 1:
        normalized["artifact_verdict"] = "borderline"
    return normalized

def aggregate_artifact_reports(reports: list[dict]) -> dict:
    if not reports:
        raise ValueError("No successful artifact reports to aggregate.")
    threshold = max(1, len(reports) // 2 + 1)
    severe_votes = Counter()
    moderate_votes = Counter()
    summary_votes = Counter()

    for report in reports:
        for value in report.get("severe_artifacts", []):
            severe_votes[value] += 1
        for value in report.get("moderate_artifacts", []):
            moderate_votes[value] += 1
        summary = str(report.get("artifact_summary", "")).strip()
        if summary:
            summary_votes[summary] += 1

    all_labels = set(severe_votes) | set(moderate_votes)

    def _sort_key(value: str):
        return (
            severe_votes[value] + moderate_votes[value],
            severe_votes[value],
            moderate_votes[value],
            value,
        )

    severe = []
    moderate = []
    vote_breakdown = {}
    for value in sorted(all_labels, key=_sort_key, reverse=True):
        severe_count = severe_votes[value]
        moderate_count = moderate_votes[value]
        total_count = severe_count + moderate_count
        vote_breakdown[value] = {
            "severe_votes": severe_count,
            "moderate_votes": moderate_count,
            "total_votes": total_count,
        }
        if severe_count >= threshold:
            severe.append(value)
        elif total_count >= threshold:
            moderate.append(value)

    summary = summary_votes.most_common(1)[0][0] if summary_votes else ""
    aggregated = normalize_artifact_report({
        "severe_artifacts": severe,
        "moderate_artifacts": moderate,
        "artifact_summary": summary,
    })

    if not aggregated.get("artifact_summary"):
        if aggregated["artifact_verdict"] == "clean":
            aggregated["artifact_summary"] = "No AI generation artifacts detected."
        elif aggregated["artifact_verdict"] == "artifact_heavy":
            aggregated["artifact_summary"] = "Consensus detected multiple visible AI artifacts."
        else:
            aggregated["artifact_summary"] = "Consensus detected some visible AI artifacts."

    aggregated["consensus_runs"] = len(reports)
    aggregated["consensus_threshold"] = threshold
    aggregated["vote_breakdown"] = vote_breakdown
    return aggregated

def penalties_for_patterns(entries: list[str], patterns: tuple[str, ...], per_match: float) -> float:
    total = 0.0
    for entry in entries:
        if any(pattern in entry for pattern in patterns):
            total += per_match
    return total

def score_from_artifact_report(video_path: str, artifact_report: dict) -> dict:
    severe = artifact_report.get("severe_artifacts", [])
    moderate = artifact_report.get("moderate_artifacts", [])
    severe_count = len(severe)
    moderate_count = len(moderate)

    identity_severe = penalties_for_patterns(severe, ("identity",), 1.0)
    identity_moderate = penalties_for_patterns(moderate, ("identity",), 1.0)
    face_severe = penalties_for_patterns(severe, ("face", "skin"), 1.0)
    face_moderate = penalties_for_patterns(moderate, ("face", "skin"), 1.0)
    texture_severe = penalties_for_patterns(severe, ("texture", "geometry", "warp"), 1.0)
    texture_moderate = penalties_for_patterns(moderate, ("texture", "geometry", "warp"), 1.0)
    hand_severe = penalties_for_patterns(severe, ("hand", "finger"), 1.0)
    hand_moderate = penalties_for_patterns(moderate, ("hand", "finger"), 1.0)
    mouth_severe = penalties_for_patterns(severe, ("mouth", "food", "object", "interaction"), 1.0)
    mouth_moderate = penalties_for_patterns(moderate, ("mouth", "food", "object", "interaction"), 1.0)
    motion_severe = penalties_for_patterns(severe, ("motion", "temporal"), 1.0)
    motion_moderate = penalties_for_patterns(moderate, ("motion", "temporal"), 1.0)

    total_penalty = severe_count * 2.2 + moderate_count * 1.0
    total_penalty += face_severe * 0.6 + face_moderate * 0.25
    total_penalty += identity_severe * 1.5 + identity_moderate * 1.0
    total_penalty += texture_severe * 0.4 + texture_moderate * 0.35
    total_penalty += hand_severe * 0.75 + hand_moderate * 0.5
    total_penalty += mouth_severe * 0.75 + mouth_moderate * 0.5
    total_penalty += motion_severe * 0.75 + motion_moderate * 0.5

    face_realism = clamp(9.5 - face_severe * 3.0 - face_moderate * 1.5)
    hand_realism = clamp(9.0 - hand_severe * 4.0 - hand_moderate * 2.0)
    mouth_food_interaction = clamp(9.0 - mouth_severe * 4.0 - mouth_moderate * 2.0)
    identity_consistency = clamp(9.5 - identity_severe * 4.5 - identity_moderate * 2.5)
    motion_realism = clamp(9.0 - motion_severe * 2.5 - motion_moderate * 1.25 - texture_severe * 1.0 - texture_moderate * 0.75)
    artifact_cleanliness = clamp(9.5 - total_penalty)
    naturalness = clamp((face_realism + hand_realism + mouth_food_interaction + identity_consistency + motion_realism + artifact_cleanliness) / 6.0)

    has_identity_issue = (identity_severe + identity_moderate) > 0

    if severe_count >= 2 or total_penalty >= 8.0:
        realism_label = "low_realism"
        confidence = clamp(6.5 + severe_count + 0.5 * moderate_count, 0, 10)
    elif (severe_count == 1 and has_identity_issue) or moderate_count >= 2 or total_penalty >= 6.0:
        realism_label = "medium_realism"
        confidence = clamp(5.0 + 0.5 * severe_count + 0.5 * moderate_count, 0, 10)
    else:
        realism_label = "high_realism"
        confidence = clamp(5.0 + max(0.0, 3.5 - total_penalty), 0, 10)

    return {
        "video": video_path,
        "origin_verdict": "ai_generated",
        "realism_label": realism_label,
        "naturalness": naturalness,
        "face_realism": face_realism,
        "hand_realism": hand_realism,
        "mouth_food_interaction": mouth_food_interaction,
        "identity_consistency": identity_consistency,
        "motion_realism": motion_realism,
        "artifact_cleanliness": artifact_cleanliness,
        "realism_score": round((naturalness + artifact_cleanliness + face_realism + identity_consistency) / 4.0, 3),
        "confidence": confidence,
        "reason": artifact_report.get("artifact_summary") or "Artifact-based realism assessment",
        "artifact_evidence": (severe[:2] + moderate[:2])[:3],
        "artifact_report": artifact_report,
    }

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
)
pipe = pipeline(
    task="any-to-any",
    model=MODEL_ID,
    device_map="auto",
    dtype=torch.float16,
    model_kwargs={
        "quantization_config": bnb_config,
        "low_cpu_mem_usage": True,
    },
)
generation_config = GenerationConfig.from_pretrained(MODEL_ID)
generation_config.max_new_tokens = MAX_NEW_TOKENS
retry_generation_config = GenerationConfig.from_pretrained(MODEL_ID)
retry_generation_config.max_new_tokens = RETRY_MAX_NEW_TOKENS


In [ ]:
ranked = []
raw_outputs = {}

def infer_json(primary_messages, retry_messages, label):
    output = pipe(
        text=primary_messages,
        return_full_text=False,
        generate_kwargs={"generation_config": generation_config},
    )
    generated_text = str(output[0]["generated_text"])
    raw_outputs[label] = generated_text
    try:
        return extract_json_object(generated_text)
    except Exception:
        retry_output = pipe(
            text=retry_messages,
            return_full_text=False,
            generate_kwargs={"generation_config": retry_generation_config},
        )
        retry_text = str(retry_output[0]["generated_text"])
        raw_outputs[f"{label}_retry"] = retry_text
        return extract_json_object(retry_text)

def collect_consensus_artifact_report(video_path: str, frame_paths: list[str], metadata: dict):
    trial_reports = []
    attempt_count = 0
    while len(trial_reports) < CONSENSUS_RUNS and attempt_count < CONSENSUS_MAX_ATTEMPTS:
        attempt_count += 1
        label = f"{Path(video_path).name}_artifact_attempt_{attempt_count}"
        artifact_primary = build_artifact_messages(video_path, frame_paths, metadata)
        artifact_retry = build_artifact_messages(video_path, frame_paths, metadata)
        try:
            artifact_report = normalize_artifact_report(
                infer_json(artifact_primary, artifact_retry, label)
            )
            trial_reports.append(artifact_report)
        except Exception as exc:
            raw_outputs[f"{label}_error"] = str(exc)
        finally:
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    if not trial_reports:
        raise RuntimeError(f"No successful artifact reports collected for {Path(video_path).name}")

    aggregated = aggregate_artifact_reports(trial_reports)
    aggregated["consensus_attempts"] = attempt_count
    return aggregated, trial_reports

for video_path in VIDEO_PATHS:
    metadata = collect_video_metadata(video_path)
    frame_paths = extract_frame_evidence(video_path)

    artifact_report, artifact_trials = collect_consensus_artifact_report(video_path, frame_paths, metadata)

    parsed = score_from_artifact_report(video_path, artifact_report)
    parsed["frame_paths"] = frame_paths
    parsed["metadata"] = metadata
    parsed["artifact_trials"] = artifact_trials
    ranked.append(parsed)
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

ranked = sorted(
    ranked,
    key=lambda item: (
        REALISM_PRIORITY.get(item.get("realism_label"), -1),
        float(item["realism_score"]),
    ),
    reverse=True,
)
for index, item in enumerate(ranked, start=1):
    item["rank"] = index

winner = ranked[0]
runner_up = ranked[1] if len(ranked) > 1 else None
summary = {
    "most_natural": {
        "video": Path(winner["video"]).name,
        "realism_score": winner["realism_score"],
        "origin_verdict": winner.get("origin_verdict"),
        "realism_label": winner.get("realism_label"),
        "confidence": winner.get("confidence"),
        "reason": winner.get("reason"),
        "artifact_evidence": winner.get("artifact_evidence", []),
        "artifact_report": winner.get("artifact_report", {}),
    },
    "less_natural": {
        "video": Path(runner_up["video"]).name,
        "realism_score": runner_up["realism_score"],
        "origin_verdict": runner_up.get("origin_verdict"),
        "realism_label": runner_up.get("realism_label"),
        "confidence": runner_up.get("confidence"),
        "reason": runner_up.get("reason"),
        "artifact_report": runner_up.get("artifact_report", {}),
    } if runner_up else None,
    "margin": round(float(winner["realism_score"]) - float(runner_up["realism_score"]), 3) if runner_up else 0.0,
    "score_table": [
        {
            "rank": item["rank"],
            "video": Path(item["video"]).name,
            "realism_score": item["realism_score"],
            "origin_verdict": item.get("origin_verdict"),
            "realism_label": item.get("realism_label"),
            "confidence": item.get("confidence"),
            "reason": item.get("reason"),
            "artifact_verdict": item.get("artifact_report", {}).get("artifact_verdict"),
            "consensus_runs": item.get("artifact_report", {}).get("consensus_runs"),
        }
        for item in ranked
    ],
}
summary


In [ ]:
print("AI-Generated Video Realism Evaluation")
print()
print(f"Most realistic AI video: {summary['most_natural']['video']}")
print(f"Origin verdict: {summary['most_natural']['origin_verdict']}")
print(f"Realism score: {summary['most_natural']['realism_score']}")
print(f"Realism label: {summary['most_natural']['realism_label']}")
print(f"Artifact verdict: {summary['most_natural']['artifact_report'].get('artifact_verdict')}")
print(f"Confidence: {summary['most_natural']['confidence']}")
print(f"Reason: {summary['most_natural']['reason']}")
if summary['most_natural']['artifact_evidence']:
    print(f"Artifact evidence: {', '.join(summary['most_natural']['artifact_evidence'][:3])}")
if summary['most_natural']['artifact_report'].get('severe_artifacts'):
    print(f"Severe artifacts: {', '.join(summary['most_natural']['artifact_report'].get('severe_artifacts', [])[:3])}")
if summary['most_natural']['artifact_report'].get('moderate_artifacts'):
    print(f"Moderate artifacts: {', '.join(summary['most_natural']['artifact_report'].get('moderate_artifacts', [])[:3])}")
print()
if summary['less_natural']:
    print(f"Less realistic AI video: {summary['less_natural']['video']}")
    print(f"Origin verdict: {summary['less_natural']['origin_verdict']}")
    print(f"Realism score: {summary['less_natural']['realism_score']}")
    print(f"Realism label: {summary['less_natural']['realism_label']}")
    print(f"Artifact verdict: {summary['less_natural']['artifact_report'].get('artifact_verdict')}")
    print(f"Confidence: {summary['less_natural']['confidence']}")
    print(f"Reason: {summary['less_natural']['reason']}")
    if summary['less_natural']['artifact_report'].get('severe_artifacts'):
        print(f"Severe artifacts: {', '.join(summary['less_natural']['artifact_report'].get('severe_artifacts', [])[:3])}")
    if summary['less_natural']['artifact_report'].get('moderate_artifacts'):
        print(f"Moderate artifacts: {', '.join(summary['less_natural']['artifact_report'].get('moderate_artifacts', [])[:3])}")
    print()
print(f"Realism margin: {summary['margin']}")


In [ ]:
summary['score_table']

In [ ]:
print(json.dumps(ranked, indent=2))